In [1]:
import os
import sys
import time
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"]        = sys.executable


In [2]:
from pyspark import SparkContext
from sympy import *
from drudge import *
from gristmill import *


In [3]:
import re

# ----------------------------------------------------------------------------
# 1) Start Spark & BCS quasiparticle drudge
# ----------------------------------------------------------------------------
ctx = SparkContext('local[*]', 'bcs_ccsdtq')
dr  = ReducedBCSDrudge(ctx)
dr.full_simplify = True
print("finished")

26/05/12 17:23:17 WARN Utils: Your hostname, Swarnamoys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.87 instead (on interface en0)
26/05/12 17:23:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 17:23:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

finished


In [4]:
p,q,r,s,i,j,k,l = dr.names.A_dumms[:8]
u, v = IndexedBase('u'), IndexedBase('v')

In [5]:
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
P_i_dag = (u[p]*v[p]+u[p]**2*Pdag[p]
     - u[p]*v[p]*N[p]
     - v[p]**2*P[p])

P_i_dag_i = (u[p+r]*v[p+r]+u[p+r]**2*Pdag[p+r]
     - u[p+r]*v[p+r]*N[p+r]
     - v[p+r]**2*P[p+r])

P_j_dag = (
        u[q]*v[q]
      + u[q]**2 * Pdag[q]
      - u[q]*v[q] * N[q]
      - v[q]**2 * P[q]
    )

P_k_dag = (
        u[r]*v[r]
      + u[r]**2 * Pdag[r]
      - u[r]*v[r] * N[r]
      - v[r]**2 * P[r]
    )
P_i = (
        (u[p])*(v[p])
      + (u[p])**2 * P[p]
      - (u[p])*(v[p]) * N[p]
      - (v[p])**2 * Pdag[p]
    )

P_i_i = (
        (u[p+r])*(v[p+r])
      + (u[p+r])**2 * P[p+r]
      - (u[p+r])*(v[p+r]) * N[p+r]
      - (v[p+r])**2 * Pdag[p+r]
    )

P_j = ((u[q])*(v[q])
      + (u[q])**2 * P[q]
      - (u[q])*(v[q]) * N[q]
      - (v[q])**2 * Pdag[q])

P_k = ((u[r])*(v[r])
      + (u[r])**2 * P[r]
      - (u[r])*(v[r]) * N[r]
      - (v[r])**2 * Pdag[r])
#hc = P_j_dag * P_i
N_i = 2*(v[p])*v[p] +((u[p])*u[p] - (v[p])*v[p])*N[p] +2*u[p]*(v[p])*Pdag[p]\
            +2*(u[p])*v[p]*P[p]
N_i_r = 2*(v[p+r])*v[p+r] +((u[p+r])*u[p+r] - (v[p+r])*v[p+r])*N[p+r] +2*u[p+r]*(v[p+r])*Pdag[p+r]\
            +2*(u[p+r])*v[p+r]*P[p+r]
N_j = 2*(v[q])*v[q] +((u[q])*u[q] - (v[q])*v[q])*N[q] +2*u[q]*(v[q])*Pdag[q]\
            +2*(u[q])*v[q]*P[q]

N_k = 2*(v[r])*v[r] +((u[r])*u[r] - (v[r])*v[r])*N[r] +2*u[r]*(v[r])*Pdag[r]\
       +2*(u[r])*v[r]*P[r]

In [6]:
def drop_terms_containing_P(expr):

    ''' Assuming expr is normal_ordered '''
    kept = expr.simplify().filter(lambda term: all(v.label != 'P' for v in term.vecs))
     
    #return Tensor(dr,kept).simplify().merge()
    return kept

def keep_exactly_k_Pdag(expr, k ,pdag_label='P^\\dagger'):

    """
    Keep only terms with exactly k creators P^\\dagger in the operator string.
    Assumes expr has already had all P (annihilators) filtered out and is normal ordered.
    """
    def nPdag(term):
        return sum(1 for v in term.vecs if v.label == pdag_label)

    kept = expr.simplify().filter(lambda term: nPdag(term) == k)
    #return Tensor(dr, kept).simplify().merge()
    return kept

In [7]:
H11, H20 , H02 = IndexedBase('H11'), IndexedBase('H20'), IndexedBase('H02')
H04, H40 = IndexedBase('H04'), IndexedBase('H40')
H22, Hb22 = IndexedBase('H22'), IndexedBase('HT22')
H31, H13 = IndexedBase('H31'), IndexedBase('H13')

In [8]:
expr = H11[p]*N[p] + H02[p]*Pdag[p] +H20[p]*P[p]+H22[p,q]*N[p]*N[q]+Hb22[p,q]*Pdag[p]*P[q]\
    +H40[p,q]*P[p]*P[q]+H04[p,q]*Pdag[p]*Pdag[q]+H13[p,q]*Pdag[p]*N[q]+H31[p,q]*N[p]*P[q]
H_N = dr.einst(expr).simplify().merge()

In [9]:
H_N.display()

<IPython.core.display.Math object>

In [10]:
t1, t2, t3,t4 = IndexedBase('t1'), IndexedBase('t2'), IndexedBase('t3'), IndexedBase('t4')

z1, z2 = IndexedBase('z1'), IndexedBase('z2')
z3, z4 = IndexedBase('z3'), IndexedBase('z4')

Z1 = dr.einst(z1[p]*P[p])
Z2 = dr.einst(Rational(1,2)*z2[p,q]*P[p]*P[q])
Z3 = dr.einst(Rational(1,6)*z3[p,q,r]*P[p]*P[q]*P[r])
Z4 = dr.einst(Rational(1,24)*z4[p,q,r,s]*P[p]*P[q]*P[r]*P[s])

T1 = dr.einst(t1[p] * Pdag[p])
T2 = dr.einst(Rational(1, 2) * t2[p,q] *Pdag[p]*Pdag[q])
T3 = dr.einst(Rational(1,6)*t3[p,q,r]*Pdag[p]*Pdag[q]*Pdag[r])
T4 = dr.einst(Rational(1,24)*t4[p,q,r,s]*Pdag[p]*Pdag[q]*Pdag[r]*Pdag[s])

Z =  Z1+Z2 #+Z3+ Z4
cluster = T1+ T2 +T3+ T4
cluster.display()

<IPython.core.display.Math object>

In [11]:
dr.set_symm(t2, Perm([1, 0],IDENT), valence=2) 
dr.set_symm(z2, Perm([1, 0],IDENT), valence=2) 
dr.set_symm(t3, Perm([1,0,2],IDENT),Perm([2,1,0],IDENT), Perm([0,2,1],IDENT), valence=3)
dr.set_symm(z3, Perm([1,0,2],IDENT),Perm([2,1,0],IDENT), Perm([0,2,1],IDENT), valence=3)
dr.set_symm(t4,Perm([1,0,2,3],IDENT),Perm([0,1,3,2],IDENT),Perm([0,2,1,3],IDENT))
dr.set_symm(z4,Perm([1,0,2,3],IDENT),Perm([0,1,3,2],IDENT),Perm([0,2,1,3],IDENT))
dr.set_symm(H04,Perm([1,0],IDENT))
dr.set_symm(H40,Perm([1,0],IDENT)) 

In [12]:
zero_term = [(H40[p,p],0),(H04[p,p],0),(t2[p,p],0),(t4[p,p,p,p],0),(t3[p,p,p],0),(t3[p,p,q],0),(t3[p,q,p],0),(z3[p,p,p],0),(z3[q,p,p],0),(z3[p,p,q],0),(z2[p,p],0),(z4[p,p,p,p],0),
    (t3[q,p,p],0),(t3[p,q,p],0),(t4[p,p,r,s],0),(t4[r,p,p,s],0),(t4[r,s,p,p],0),(t4[p,r,p,s],0),(t4[p,r,s,p],0),(t4[r,p,s,p],0),
            (z4[p,p,r,s],0),(z4[r,p,p,s],0),(z4[r,s,p,p],0),(z4[p,r,p,s],0),(z4[p,r,s,p],0),(z4[r,p,s,p],0)]

In [13]:
t0 = time.time()
Hbar = H_N
curr = H_N
fact = 1
for n in range(5):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    Hbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
print("Hbar evaluation done",time.time()-t0)

[Stage 153:==================>                                     (4 + 8) / 12]

Hbar evaluation done 70.56701278686523


In [14]:
Hbar_noP = drop_terms_containing_P(Hbar)

In [15]:
E_eqn  = Hbar_noP.simplify().eval_vev().simplify()
E_eqn = E_eqn.subst_all(zero_term)

In [16]:
print(E_eqn.latex())

\sum_{p \in A} \sum_{q \in A} H^{(40)}_{p,q}  t^{(2)}_{p,q}   + \sum_{p \in A} \sum_{q \in A} t^{(1)}_{p}  t^{(1)}_{q}  H^{(40)}_{p,q}   + \sum_{p \in A} H^{(20)}_{p}  t^{(1)}_{p} 


In [17]:
#T1 equations

Hbar_T1 = keep_exactly_k_Pdag(Hbar_noP,1)
t0 = time.time()
s1_eqn_CCSDTQ = (P[p] * Hbar_T1).simplify().eval_vev().simplify()
print('evaluation done', time.time()-t0) 

evaluation done 5.8066041469573975


In [18]:
s1_eqn_CCSDTQ = s1_eqn_CCSDTQ.subst_all(zero_term)
print(s1_eqn_CCSDTQ.latex())

\sum_{q \in A} H^{(20)}_{q}  t^{(2)}_{q,p}   + \sum_{q \in A} t^{(1)}_{q}  HT^{(22)}_{p,q}   + \sum_{q \in A} 2 H^{(31)}_{p,q}  t^{(2)}_{q,p}  - \sum_{q \in A} 4 t^{(1)}_{p}  H^{(40)}_{q,p}  t^{(2)}_{q,p}  - \sum_{q \in A} 2 t^{(1)}_{p}^{2}  t^{(1)}_{q}  H^{(40)}_{q,p}   + \sum_{q \in A} 2 t^{(1)}_{p}  t^{(1)}_{q}  H^{(31)}_{p,q}  - t^{(1)}_{p}^{2}  H^{(20)}_{p}  - 2 t^{(1)}_{p}^{2}  H^{(31)}_{p,p}   + 2 H^{(11)}_{p}  t^{(1)}_{p}   + 4 t^{(1)}_{p}  H^{(22)}_{p,p}   + H^{(02)}_{p}   + \sum_{q \in A} \sum_{r \in A} H^{(40)}_{q,r}  t^{(3)}_{q,r,p}   + \sum_{q \in A} \sum_{r \in A} 2 t^{(1)}_{q}  H^{(40)}_{q,r}  t^{(2)}_{r,p} 


In [19]:
#T2 equations

Hbar_T2 = keep_exactly_k_Pdag(Hbar_noP,2)
t0 = time.time()
s2_eqn_CCSDTQ = (P[p]*P[q] * Hbar_T2).simplify().eval_vev().simplify()
s2_eqn_CCSDTQ = s2_eqn_CCSDTQ.subst_all(zero_term).simplify()
print('evaluation done in ',time.time()-t0)
print(s2_eqn_CCSDTQ.latex())

evaluation done in  32.260597229003906
\sum_{r \in A} \sum_{s \in A} H^{(40)}_{r,s}  t^{(4)}_{r,s,p,q}   + \sum_{r \in A} \sum_{s \in A} 2 t^{(1)}_{r}  H^{(40)}_{r,s}  t^{(3)}_{s,p,q}   + \sum_{r \in A} \sum_{s \in A} 2 H^{(40)}_{r,s}  t^{(2)}_{r,p}  t^{(2)}_{s,q}  - \sum_{r \in A} \sum_{s \in A}  2 \delta_{p q} H^{(40)}_{r,s}  t^{(2)}_{r,p}  t^{(2)}_{s,p}   + \sum_{r \in A} H^{(20)}_{r}  t^{(3)}_{r,p,q}   + \sum_{r \in A} HT^{(22)}_{p,r}  t^{(2)}_{r,q}   + \sum_{r \in A} HT^{(22)}_{q,r}  t^{(2)}_{r,p}   + \sum_{r \in A} 2 H^{(31)}_{p,r}  t^{(3)}_{r,p,q}   + \sum_{r \in A} 2 H^{(31)}_{q,r}  t^{(3)}_{r,p,q}  - \sum_{r \in A} 4 t^{(1)}_{p}  H^{(40)}_{r,p}  t^{(3)}_{r,p,q}  - \sum_{r \in A} 4 t^{(1)}_{q}  H^{(40)}_{r,q}  t^{(3)}_{r,p,q}  - \sum_{r \in A} 4 H^{(40)}_{r,p}  t^{(2)}_{p,q}  t^{(2)}_{r,p}  - \sum_{r \in A} 4 H^{(40)}_{r,q}  t^{(2)}_{p,q}  t^{(2)}_{r,q}  - \sum_{r \in A} 2 t^{(1)}_{p}^{2}  H^{(40)}_{r,p}  t^{(2)}_{r,q}  - \sum_{r \in A} 2 t^{(1)}_{q}^{2}  H^{(40)}_{r,q}  t^{(2)

In [24]:
#T3 equations

Hbar_T3 = keep_exactly_k_Pdag(Hbar_noP,3)
t0 = time.time()
s3_eqn_CCSDTQ = (P[p]*P[q]*P[r] * Hbar_T3).simplify().eval_vev().simplify()
s3_eqn_CCSDTQ = s3_eqn_CCSDTQ.subst_all(zero_term)
print('evaluation done in ', time.time()-t0)

evaluation done in  290.74122405052185


In [22]:
#T4 


Hbar_T4 = keep_exactly_k_Pdag(Hbar_noP,4)
t0 = time.time()
s4_eqn_CCSDTQ = (P[p]*P[q]*P[r]*P[s] * Hbar_T4).simplify().eval_vev().simplify()
s4_eqn_CCSDTQ = s4_eqn_CCSDTQ.subst_all(zero_term)
print('evaluation done in ', time.time()-t0)

evaluation done in  7393.577898979187


In [25]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
Ene = IndexedBase('Ene')
expr0 = dr.define(Ene,E_eqn)
Res1 = IndexedBase('Res1')
expr1 = dr.define(Res1[p],s1_eqn_CCSDTQ)
Res2 = IndexedBase('Res2')
expr2 = dr.define(Res2[p,q],s2_eqn_CCSDTQ)
Res3 = IndexedBase('Res3')
expr3 = dr.define(Res3[p,q,r],s3_eqn_CCSDTQ)
Res4 = IndexedBase('Res4')
expr4 = dr.define(Res4[p,q,r,s],s4_eqn_CCSDTQ)

org_exp = [expr0,expr1,expr2,expr3,expr4]
print("Original Cost:", get_flop_cost(org_exp))
eval_equ0 = optimize(
    [expr0,expr1,expr2,expr3,expr4],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
 
print("Optimized cost:", get_flop_cost(eval_equ0))

 
fort = FortranPrinter()
code = fort.doprint(eval_equ0)
#print(code)


delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("CCRes_CCSDTQ.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(code)



Original Cost: 84*na**6 + 1306*na**5 + 3687*na**4 + 587*na**3 + 115*na**2 + 18*na + 2


/Users/swarnamoyghosh/anaconda3/envs/drudge_fix/lib/python3.9/site-packages/gristmill/optimize.py:1984: UserWarning: Internal deficiency: summation intermediate may not be fully canonicalized
  warnings.warn(


KeyboardInterrupt: 